# 01 — Análise exploratória: hipóteses que viram decisões de modelagem

Cada seção termina com uma **decisão** (manter / remover / transformar uma feature, ou uma escolha de desenho). Nenhuma lógica vive aqui: tudo vem de `src/`. Escalas: taxas da Gold em pontos percentuais (0–100); externas na escala de origem (ver `docs/dicionario_base_modelagem.md`).

In [1]:
import sys; sys.path.insert(0, "..")
import numpy as np
import pandas as pd
from scipy.stats import spearmanr

from src import config
from src.preprocessing import contexto, features
from src.preprocessing.carregar import carregar_alunos
from src.visualization import plots

pd.set_option("display.width", 160)
aluno = pd.read_parquet(config.PROCESSED / "base_modelagem_aluno.parquet")
mun = pd.read_parquet(config.PROCESSED / "base_modelagem_municipio.parquet")
m24 = mun[mun["ano"] == 2024].copy()
alunos_2024 = carregar_alunos(2024, apenas_com_nota=False)
print(aluno.shape, mun.shape, alunos_2024.shape)

C:\Users\tcarm\Projetos\predicao-alfabetiza-brasil\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


(1851828, 103) (10276, 84) (2120560, 20)


## 1. Universo e alvo

In [2]:
universo = pd.Series({
    "linhas 2024 na Silver": len(alunos_2024),
    "presentes com nota": int((alunos_2024["presente"] & ~alunos_2024["sem_nota"]).sum()),
    "rede pública com nota (base de modelagem)": len(aluno),
    "alfabetizados (%)": round(100 * aluno["alfabetizado"].mean(), 1),
    "municípios": aluno["id_municipio"].nunique(),
    "escolas": aluno["id_escola"].nunique(),
    "alunos em município sem histórico 2023 (%)": round(100 * aluno["sem_historico"].mean(), 1),
})
display(universo.to_frame("valor"))
plots.salvar(plots.plot_distribuicao_proficiencia(aluno["proficiencia"]), "eda_distribuicao_proficiencia")

,valor
linhas 2024 na Silver,2120560.0
presentes com nota,1851852.0
rede pública com nota (base de modelagem),1851828.0
alfabetizados (%),59.8
municípios,5517.0
escolas,42327.0
alunos em município sem histórico 2023 (%),23.1


WindowsPath('C:/Users/tcarm/Projetos/predicao-alfabetiza-brasil/images/eda_distribuicao_proficiencia.png')

**Decisão:** classes ≈ 60/40, sem reamostragem; métricas reportadas para a classe "não alfabetizado" (recall, precisão, PR-AUC). O universo é presentes com nota da rede pública; ausência vira análise (H6), não classe.

## H1 — Rede estadual e municipal têm taxas diferentes dentro do mesmo município

In [3]:
h1 = aluno.groupby(["id_municipio", "rede_nome"], observed=True)["alfabetizado"].mean().unstack().dropna()
h1["diferenca_pp"] = 100 * (h1["estadual"] - h1["municipal"])
print(f"municípios com as duas redes: {len(h1)}; diferença mediana estadual − municipal: {h1['diferenca_pp'].median():.1f} pp")
long = h1[["estadual", "municipal"]].mul(100).melt(var_name="rede_nome", value_name="taxa_alfabetizacao")
plots.salvar(plots.plot_boxplot_por_grupo(long, "rede_nome", "taxa_alfabetizacao", "Taxa por rede, municípios com as duas redes"), "eda_h1_rede")

municípios com as duas redes: 1018; diferença mediana estadual − municipal: 4.5 pp


WindowsPath('C:/Users/tcarm/Projetos/predicao-alfabetiza-brasil/images/eda_h1_rede.png')

**Decisão:** manter `rede_nome` como categórica (a diferença pareada existe e não é explicada pelo município).

## H2 — Alfabetização adulta e PIB per capita explicam a taxa municipal

In [4]:
for col in ["taxa_alfabetizacao_adultos", "log_pib_per_capita"]:
    d = m24[[col, "taxa_alfabetizacao"]].dropna()
    print(f"{col}: Spearman = {spearmanr(d[col], d['taxa_alfabetizacao']).statistic:.3f}")
por_regiao = m24.groupby("regiao").apply(lambda g: spearmanr(g["taxa_alfabetizacao_adultos"], g["taxa_alfabetizacao"], nan_policy="omit").statistic).round(3)
display(por_regiao.to_frame("spearman_alfabetizacao_adultos"))
plots.salvar(plots.plot_dispersao(m24, "taxa_alfabetizacao_adultos", "taxa_alfabetizacao", hue="regiao"), "eda_h2_socioeconomico")

taxa_alfabetizacao_adultos: Spearman = 0.119
log_pib_per_capita: Spearman = 0.116


C:\Users\tcarm\AppData\Local\Temp\ipykernel_18760\567418570.py:4: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  por_regiao = m24.groupby("regiao").apply(lambda g: spearmanr(g["taxa_alfabetizacao_adultos"], g["taxa_alfabetizacao"], nan_policy="omit").statistic).round(3)


,spearman_alfabetizacao_adultos
regiao,
Centro-Oeste,-0.240
Nordeste,-0.155
Norte,0.210
Sudeste,-0.257
Sul,-0.222


WindowsPath('C:/Users/tcarm/Projetos/predicao-alfabetiza-brasil/images/eda_h2_socioeconomico.png')

**Decisão:** manter as socioeconômicas; PIB per capita entra em log (`log_pib_per_capita`), a relação é côncava. Achado: o Spearman nacional é fraco (≈0,12) e mascara heterogeneidade regional — dentro de 4 das 5 regiões a correlação com alfabetização adulta é negativa (só o Norte é positiva); a leitura fica para o SHAP por região no notebook 02, não para um coeficiente nacional.

## H3 — Infraestrutura escolar do município se associa à taxa

In [5]:
infra = ["pct_escolas_internet", "pct_escolas_biblioteca", "pct_escolas_esgoto_rede", "pct_escolas_agua_potavel", "pct_escolas_lab_informatica"]
print(m24[infra + ["taxa_alfabetizacao"]].corr(method="spearman")["taxa_alfabetizacao"].drop("taxa_alfabetizacao").round(3))
q = m24.assign(quartil_internet=pd.qcut(m24["pct_escolas_internet"].rank(method="first"), 4, labels=["Q1", "Q2", "Q3", "Q4"]))
plots.salvar(plots.plot_boxplot_por_grupo(q, "quartil_internet", "taxa_alfabetizacao", "Taxa por quartil de escolas com internet"), "eda_h3_infraestrutura")

pct_escolas_internet           0.224
pct_escolas_biblioteca         0.257
pct_escolas_esgoto_rede        0.129
pct_escolas_agua_potavel       0.196
pct_escolas_lab_informatica    0.145
Name: taxa_alfabetizacao, dtype: float64


WindowsPath('C:/Users/tcarm/Projetos/predicao-alfabetiza-brasil/images/eda_h3_infraestrutura.png')

**Decisão:** manter os agregados de infraestrutura; os de correlação mais alta ficam para o SHAP dizer se acrescentam além do socioeconômico.

## H4 — Distorção idade-série nos anos iniciais antecipa não alfabetização

In [6]:
d = m24[["tdi_ai", "taxa_alfabetizacao"]].dropna()
print(f"Spearman tdi_ai × taxa: {spearmanr(d['tdi_ai'], d['taxa_alfabetizacao']).statistic:.3f} (n={len(d)})")
plots.salvar(plots.plot_dispersao(m24, "tdi_ai", "taxa_alfabetizacao", hue="regiao"), "eda_h4_tdi")

Spearman tdi_ai × taxa: -0.079 (n=5450)


WindowsPath('C:/Users/tcarm/Projetos/predicao-alfabetiza-brasil/images/eda_h4_tdi.png')

**Decisão:** manter `tdi_ai` (indicador educacional complementar, t−1).

## H5 — Porte e ruralidade: municípios pequenos e rurais têm taxa menor e mais incerta

In [7]:
for col in ["log_populacao", "pct_escolas_rurais"]:
    d = m24[[col, "taxa_alfabetizacao"]].dropna()
    print(f"{col}: Spearman = {spearmanr(d[col], d['taxa_alfabetizacao']).statistic:.3f}")
porte = m24.assign(porte=pd.qcut(m24["alunos_com_nota"], 4, labels=["<Q1", "Q1–Q2", "Q2–Q3", ">Q3"]))
print(porte.groupby("porte", observed=True)["ic95"].median().round(1).rename("ic95 mediana (pp)"))
plots.salvar(plots.plot_boxplot_por_grupo(porte, "porte", "ic95", "Margem de erro (ic95) por porte do município"), "eda_h5_porte")

log_populacao: Spearman = -0.191
pct_escolas_rurais: Spearman = -0.233
porte
<Q1      15.1
Q1–Q2    10.2
Q2–Q3     7.2
>Q3       4.2
Name: ic95 mediana (pp), dtype: float64


WindowsPath('C:/Users/tcarm/Projetos/predicao-alfabetiza-brasil/images/eda_h5_porte.png')

**Decisão:** manter `log_populacao`, `pct_escolas_rurais` e `ic95_mun_t1` (a incerteza do histórico é informação). Alerta: rankings por taxa sem `ic95` enchem as pontas com ruído.

## H6 — A ausência não é aleatória

In [8]:
d = m24[["taxa_participacao", "taxa_alfabetizacao"]].dropna()
print(f"Spearman participação × taxa: {spearmanr(d['taxa_participacao'], d['taxa_alfabetizacao']).statistic:.3f}")
pub = alunos_2024[alunos_2024["rede_nome"].isin(["municipal", "estadual"])]
print(pub.groupby("rede_nome", observed=True)["presente"].mean().mul(100).round(1).rename("presença (%)"))
print(pub.groupby("sigla_uf", observed=True)["presente"].mean().mul(100).round(1).sort_values().head(5).rename("presença (%) — 5 menores"))
plots.salvar(plots.plot_dispersao(m24, "taxa_participacao", "taxa_alfabetizacao", hue="regiao"), "eda_h6_ausencia")

Spearman participação × taxa: 0.290


rede_nome
estadual     86.1
municipal    87.6
Name: presença (%), dtype: float64
sigla_uf
SC    70.1
RN    77.7
AM    79.6
DF    79.8
AC    80.9
Name: presença (%) — 5 menores, dtype: float64


WindowsPath('C:/Users/tcarm/Projetos/predicao-alfabetiza-brasil/images/eda_h6_ausencia.png')

**Decisão:** universo = presentes com nota; `taxa_participacao_mun_t1` entra como feature (a ausência de ontem informa o contexto de hoje). A fonte codifica ausente como 0, o que seria rótulo falso.

## H7 — Gradiente regional

In [9]:
print(m24.groupby("regiao")["taxa_alfabetizacao"].median().round(1).sort_values())
plots.salvar(plots.plot_boxplot_por_grupo(m24, "sigla_uf", "taxa_alfabetizacao", "Taxa de alfabetização por UF (2024)"), "eda_h7_regiao")

regiao
Norte           49.2
Nordeste        52.6
Sul             66.1
Sudeste         70.6
Centro-Oeste    73.4
Name: taxa_alfabetizacao, dtype: float64


WindowsPath('C:/Users/tcarm/Projetos/predicao-alfabetiza-brasil/images/eda_h7_regiao.png')

**Decisão:** `regiao` e `sigla_uf` categóricas. Conferir no SHAP (notebook 02) se as socioeconômicas absorvem o efeito regional.

## H8 — A forma da distribuição carrega informação além da média

In [10]:
faixa = m24[m24["taxa_alfabetizacao"].between(58, 62)].copy()
faixa["grupo"] = np.where(faixa["pct_critico"] >= faixa["pct_critico"].median(), "mais cauda crítica", "menos cauda crítica")
print(f"{len(faixa)} municípios com taxa entre 58 e 62 pp; pct_critico varia de {faixa['pct_critico'].min():.1f} a {faixa['pct_critico'].max():.1f}")
plots.salvar(plots.plot_perfis_niveis(faixa, "grupo"), "eda_h8_forma")

376 municípios com taxa entre 58 e 62 pp; pct_critico varia de 0.0 a 33.1


WindowsPath('C:/Users/tcarm/Projetos/predicao-alfabetiza-brasil/images/eda_h8_forma.png')

**Decisão:** `pct_nivel_0..8_mun_t1`, `pct_critico`, `pct_atencao`, `pct_quase_la` entram no modelo A; `pct_nivel_*` são a base dos clusters (notebook 04).

## H9 — O que a escola explica além do município

In [11]:
com_nota = pub[pub["presente"] & ~pub["sem_nota"]]
display(contexto.decompor_variancia(com_nota).round(2))
d = aluno.dropna(subset=["taxa_escola_loo"])
agg = d.groupby(pd.cut(d["taxa_escola_loo"], 10), observed=True)["alfabetizado"].mean().mul(100).rename("taxa_observada").reset_index()
agg["faixa"] = agg["taxa_escola_loo"].astype(str)
plots.salvar(plots.plot_barras_horizontais(agg, "taxa_observada", coluna_nome="faixa", top=10, titulo="Taxa do aluno por faixa de taxa da escola (leave-one-out)"), "eda_h9_escola")

,componente,variancia,participacao_pct
0,entre_municipios,353.3,15.96
1,entre_escolas,199.3,9.01
2,intra_escola,1660.5,75.03


WindowsPath('C:/Users/tcarm/Projetos/predicao-alfabetiza-brasil/images/eda_h9_escola.png')

**Decisão:** 75% da variância é intra-escola: o teto do modelo é baixo por construção. O contexto da escola no mesmo ano só entra no regime **diagnóstico** (leave-one-out), nunca no de produção.

## H10 — `caderno` é artefato do instrumento, não característica da criança

In [12]:
h10 = com_nota.groupby("caderno", observed=True)["alfabetizado"].agg(["mean", "size"]).rename(columns={"mean": "taxa", "size": "n"})
h10["taxa"] = 100 * h10["taxa"]
display(h10.round(1))
plots.salvar(plots.plot_barras_horizontais(h10.reset_index(), "taxa", coluna_nome="caderno", top=len(h10), titulo="Taxa por caderno de prova"), "eda_h10_caderno")

,taxa,n
caderno,,
1,59.6,97138
10,60.3,85391
11,59.8,85200
12,60.5,85278
13,60.4,85242
14,60.2,84971
15,59.1,86462
16,59.7,85207
17,59.2,85194


WindowsPath('C:/Users/tcarm/Projetos/predicao-alfabetiza-brasil/images/eda_h10_caderno.png')

**Decisão:** `caderno` fica fora do modelo (`COLUNAS_PROIBIDAS`). Diferença entre cadernos, se houver, é achado sobre equivalência dos instrumentos, não feature.

## H11 — Bolsa Família: sinal nacional e heterogeneidade por UF

In [13]:
d = m24[["pct_familias_bolsa_familia", "taxa_alfabetizacao", "sigla_uf"]].dropna()
print(f"Spearman nacional: {spearmanr(d['pct_familias_bolsa_familia'], d['taxa_alfabetizacao']).statistic:.3f}")
por_uf = d.groupby("sigla_uf").apply(lambda g: spearmanr(g["pct_familias_bolsa_familia"], g["taxa_alfabetizacao"]).statistic if len(g) >= 30 else np.nan).dropna().round(2).sort_values()
display(por_uf.to_frame("spearman_por_uf").T)
plots.salvar(plots.plot_dispersao(m24, "pct_familias_bolsa_familia", "taxa_alfabetizacao", hue="regiao"), "eda_h11_bolsa_familia")

Spearman nacional: -0.260


C:\Users\tcarm\AppData\Local\Temp\ipykernel_18760\1139224422.py:3: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  por_uf = d.groupby("sigla_uf").apply(lambda g: spearmanr(g["pct_familias_bolsa_familia"], g["taxa_alfabetizacao"]).statistic if len(g) >= 30 else np.nan).dropna().round(2).sort_values()


sigla_uf,RO,TO,RN,SC,RS,SE,PA,BA,RJ,MG,...,PR,ES,SP,MS,GO,CE,MA,AM,PE,AL
spearman_por_uf,-0.48,-0.4,-0.34,-0.33,-0.32,-0.3,-0.29,-0.28,-0.26,-0.22,...,-0.14,-0.13,-0.12,-0.12,-0.08,-0.06,-0.02,-0.01,0.04,0.05


WindowsPath('C:/Users/tcarm/Projetos/predicao-alfabetiza-brasil/images/eda_h11_bolsa_familia.png')

**Decisão:** manter `pct_familias_bolsa_familia`; o sinal muda de UF para UF (positivo em parte do Nordeste), então a leitura fica com o SHAP por região (notebook 02), não com um coeficiente nacional.

## Faltantes e colinearidade

In [14]:
num, cat = features.colunas_por_regime(aluno, "producao")
faltantes = aluno[num].isna().mean().mul(100).sort_values(ascending=False)
display(faltantes[faltantes > 0].round(1).to_frame("% faltante"))
pares = features.correlacao_alta(aluno[num].sample(200_000, random_state=config.SEED), limite=0.95)
display(pares)
vif = features.calcular_vif(m24[[c for c in num if c in m24.columns]].drop(columns=[c for c in ["pct_va_agropecuaria", "pct_va_industria", "pct_va_servicos", "pct_va_adespss"] if c in m24.columns]))
display(vif.head(15))
plots.salvar(plots.plot_barras_horizontais(faltantes[faltantes > 0].reset_index().rename(columns={"index": "feature", 0: "pct"}), "pct", top=20, titulo="% faltante por feature (produção)"), "eda_faltantes")
plots.salvar(plots.plot_correlacao(m24[[c for c in num if c in m24.columns]].sample(frac=1, random_state=config.SEED).iloc[:, :30]), "eda_correlacao")

,% faltante
pct_va_agropecuaria,100.0
pct_va_industria,100.0
pct_va_servicos,100.0
pct_va_adespss,100.0
projecao_ideb_ai,100.0
pct_critico_mun_t1,23.1
pct_nivel_4_mun_t1,23.1
pct_quase_la_mun_t1,23.1
pct_atencao_mun_t1,23.1
pct_nivel_8_mun_t1,23.1


,a,b,rho
0,pop_2022,populacao,1.000000
1,domicilios_2022,populacao,0.999660
2,alunos_avaliados_mun_t1,matriculas_ai,0.999658
3,pop_2022,domicilios_2022,0.999653
4,taxa_aprovacao_ideb_ai,rendimento_ideb_ai,0.999127
5,matriculas_ai,docentes_ai,0.997917
6,pop_2022,matriculas_ai,0.996626
7,populacao,matriculas_ai,0.996615
8,pop_2022,docentes_ai,0.995482
9,populacao,docentes_ai,0.995468


,feature,vif
0,populacao,284020.639421
1,pop_2022,269648.750834
2,domicilios_2022,1366.720628
3,ideb_ai,523.811374
4,rendimento_ideb_ai,320.933741
5,taxa_aprovacao_ideb_ai,305.815870
6,matriculas_ai,181.724824
7,nota_saeb_mat_ai,142.347764
8,nota_saeb_lp_ai,100.228191
9,docentes_ai,76.380437


WindowsPath('C:/Users/tcarm/Projetos/predicao-alfabetiza-brasil/images/eda_correlacao.png')

**Decisão:** imputação pela mediana com indicador de faltante (`add_indicator=True`) dentro do pipeline; `pct_va_*` (100% faltante para o alvo 2024) vira só indicador, sem informação — documentado. A colinearidade é mais extensa do que o esperado: 36 pares com |ρ| > 0,95 (tabela acima, sobretudo população/domicílios/matrículas/docentes e os indicadores derivados do Ideb) e VIF na casa das centenas de milhar para `populacao`/`pop_2022`. Mesmo assim, os pares são mantidos: HGB é indiferente a colinearidade e a logística tem regularização L2; para leitura de coeficientes, olhar um de cada par por cluster.

## Tabela final — hipótese → decisão

| Hipótese | Resultado | Decisão de modelagem |
|---|---|---|
| H1 rede | diferença pareada estadual − municipal (mediana acima) | `rede_nome` categórica |
| H2 socioeconômico | Spearman nacional positivo mas fraco (≈0,12); negativo em 4 de 5 regiões | manter; log no PIB; leitura por região no SHAP |
| H3 infraestrutura | correlações positivas moderadas | manter agregados; SHAP decide relevância |
| H4 TDI | correlação negativa | manter `tdi_ai` |
| H5 porte/ruralidade | pequenos e rurais: taxa menor, `ic95` maior | manter porte; `ic95_mun_t1` como feature |
| H6 ausência | participação correlaciona com taxa | universo presentes com nota; `taxa_participacao_mun_t1` |
| H7 região | gradiente N/NE × S/SE | `regiao`, `sigla_uf` categóricas |
| H8 forma | mesma taxa, caudas diferentes | `pct_nivel_*` no modelo A e nos clusters |
| H9 escola | 75% intra-escola | regime diagnóstico só para interpretação; teto baixo |
| H10 caderno | artefato de instrumento | fora |
| H11 Bolsa Família | sinal nacional negativo, heterogêneo por UF | manter; leitura por região no SHAP |
| Faltantes/colinearidade | `pct_va_*` 100% NaN; 36 pares > 0,95, VIF extremo em população/matrículas/docentes | imputação + indicador; sem remoção |